# RealEstatePRO

In [39]:
#import all necessary libraries
import pandas as pd
import os
from langchain_openai import OpenAIEmbeddings
import faiss
from langchain_community.docstore.in_memory import InMemoryDocstore
from langchain_community.vectorstores import FAISS
from langchain_core.documents import Document
import tqdm

In [67]:
os.environ["OPENAI_API_KEY"] = os.getenv("OPENAI_API_KEY") \
    or getpass("Enter your OpenAI API key: ")

## DataFrame import

In [14]:
df = pd.read_csv('NY-House-Dataset.csv')

In [15]:
df.head(10)

,BROKERTITLE,TYPE,PRICE,BEDS,BATH,PROPERTYSQFT,ADDRESS,STATE,MAIN_ADDRESS,ADMINISTRATIVE_AREA_LEVEL_2,LOCALITY,SUBLOCALITY,STREET_NAME,LONG_NAME,FORMATTED_ADDRESS,LATITUDE,LONGITUDE
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856
5,Brokered by Sowae Corp,House for sale,690000,5,2.000000,4004.000000,584 Park Pl,"Brooklyn, NY 11238","584 Park PlBrooklyn, NY 11238",United States,New York,Kings County,Brooklyn,Park Place,"584 Park Pl, Brooklyn, NY 11238, USA",40.674363,-73.958725
6,Brokered by Douglas Elliman - 575 Madison Ave,Condo for sale,899500,2,2.000000,2184.207862,157 W 126th St Unit 1B,"New York, NY 10027","157 W 126th St Unit 1BNew York, NY 10027",New York,New York County,New York,Manhattan,157,"157 W 126th St #1b, New York, NY 10027, USA",40.809448,-73.946777
7,Brokered by Connie Profaci Realty,House for sale,16800000,8,16.000000,33000.000000,177 Benedict Rd,"Staten Island, NY 10304","177 Benedict RdStaten Island, NY 10304",United States,New York,Richmond County,Staten Island,Benedict Road,"177 Benedict Rd, Staten Island, NY 10304, USA",40.595002,-74.106424
8,Brokered by Pantiga Group Inc.,Co-op for sale,265000,1,1.000000,750.000000,875 Morrison Ave Apt 3M,"Bronx, NY 10473","875 Morrison Ave Apt 3MBronx, NY 10473",Bronx County,The Bronx,East Bronx,Morrison Avenue,Parking lot,"Parking lot, 875 Morrison Ave #3m, Bronx, NY 1...",40.821586,-73.874089
9,Brokered by CENTURY 21 MK Realty,Co-op for sale,440000,2,1.000000,978.000000,1350 Ocean Pkwy Apt 5G,"Brooklyn, NY 11230","1350 Ocean Pkwy Apt 5GBrooklyn, NY 11230",New York,Kings County,Brooklyn,Midwood,1350,"1350 Ocean Pkwy #5g, Brooklyn, NY 11230, USA",40.615738,-73.969694


In [27]:
df.describe()

,price,beds,bath,propertysqft,latitude,longitude
count,4.801000e+03,4801.000000,4801.000000,4801.000000,4801.000000,4801.000000
mean,2.356940e+06,3.356801,2.373861,2184.207862,40.714227,-73.941601
std,3.135525e+07,2.602315,1.946962,2377.140894,0.087676,0.101082
min,2.494000e+03,1.000000,0.000000,230.000000,40.499546,-74.253033
25%,4.990000e+05,2.000000,1.000000,1200.000000,40.639375,-73.987143
50%,8.250000e+05,3.000000,2.000000,2184.207862,40.726749,-73.949189
75%,1.495000e+06,4.000000,3.000000,2184.207862,40.771923,-73.870638
max,2.147484e+09,50.000000,50.000000,65535.000000,40.912729,-73.702450


In [19]:
df.shape

(4801, 17)

In [20]:
#make all columns lowercase
df.columns = [col.lower() for col in df.columns]

In [28]:
df.columns

Index(['brokertitle', 'type', 'price', 'beds', 'bath', 'propertysqft',
       'address', 'state', 'main_address', 'administrative_area_level_2',
       'locality', 'sublocality', 'street_name', 'long_name',
       'formatted_address', 'latitude', 'longitude'],
      dtype='object')

### Save embeddings in FAISS

What is FAISS: TODO

In [ ]:
index_path = "faiss_index_dir"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

In [ ]:
if os.path.exists(index_path):
    # Load existing FAISS index
    vector_store = FAISS.load_local(index_path, embeddings)
    print("Loaded existing FAISS index.")

In [48]:
# Create a concise textual representation for embedding
df['text_to_embed'] = (
    df['brokertitle'].astype(str) + ", " +
    df['type'].astype(str) + ", Price: " + df['price'].astype(str) + "$, " +
    "Beds: " + df['beds'].astype(str) + ", Baths: " + df['bath'].astype(str) + ", " +
    "Size: " + df['propertysqft'].astype(str) + " sqft, " +
    "Address: " + df['address'].astype(str) + ", " +
    "Locality: " + df['locality'].astype(str) + ", " +
    "State: " + df['state'].astype(str)
)

In [49]:
df

,brokertitle,type,price,beds,bath,propertysqft,address,state,main_address,administrative_area_level_2,locality,sublocality,street_name,long_name,formatted_address,latitude,longitude,text_to_embed
0,Brokered by Douglas Elliman -111 Fifth Ave,Condo for sale,315000,2,2.000000,1400.000000,2 E 55th St Unit 803,"New York, NY 10022","2 E 55th St Unit 803New York, NY 10022",New York County,New York,Manhattan,East 55th Street,Regis Residence,"Regis Residence, 2 E 55th St #803, New York, N...",40.761255,-73.974483,"Brokered by Douglas Elliman -111 Fifth Ave, C..."
1,Brokered by Serhant,Condo for sale,195000000,7,10.000000,17545.000000,Central Park Tower Penthouse-217 W 57th New Yo...,"New York, NY 10019",Central Park Tower Penthouse-217 W 57th New Yo...,United States,New York,New York County,New York,West 57th Street,"217 W 57th St, New York, NY 10019, USA",40.766393,-73.980991,"Brokered by Serhant, Condo for sale, Price: 19..."
2,Brokered by Sowae Corp,House for sale,260000,4,2.000000,2015.000000,620 Sinclair Ave,"Staten Island, NY 10312","620 Sinclair AveStaten Island, NY 10312",United States,New York,Richmond County,Staten Island,Sinclair Avenue,"620 Sinclair Ave, Staten Island, NY 10312, USA",40.541805,-74.196109,"Brokered by Sowae Corp, House for sale, Price:..."
3,Brokered by COMPASS,Condo for sale,69000,3,1.000000,445.000000,2 E 55th St Unit 908W33,"Manhattan, NY 10022","2 E 55th St Unit 908W33Manhattan, NY 10022",United States,New York,New York County,New York,East 55th Street,"2 E 55th St, New York, NY 10022, USA",40.761398,-73.974613,"Brokered by COMPASS, Condo for sale, Price: 69..."
4,Brokered by Sotheby's International Realty - E...,Townhouse for sale,55000000,7,2.373861,14175.000000,5 E 64th St,"New York, NY 10065","5 E 64th StNew York, NY 10065",United States,New York,New York County,New York,East 64th Street,"5 E 64th St, New York, NY 10065, USA",40.767224,-73.969856,Brokered by Sotheby's International Realty - E...
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4796,Brokered by COMPASS,Co-op for sale,599000,1,1.000000,2184.207862,222 E 80th St Apt 3A,"Manhattan, NY 10075","222 E 80th St Apt 3AManhattan, NY 10075",New York,New York County,New York,Manhattan,222,"222 E 80th St #3a, New York, NY 10075, USA",40.774350,-73.955879,"Brokered by COMPASS, Co-op for sale, Price: 59..."
4797,Brokered by Mjr Real Estate Llc,Co-op for sale,245000,1,1.000000,2184.207862,97-40 62 Dr Unit Lg,"Rego Park, NY 11374","97-40 62 Dr Unit LgRego Park, NY 11374",United States,New York,Queens County,Queens,62nd Drive,"97-40 62nd Dr, Rego Park, NY 11374, USA",40.732538,-73.860152,"Brokered by Mjr Real Estate Llc, Co-op for sal..."
4798,Brokered by Douglas Elliman - 575 Madison Ave,Co-op for sale,1275000,1,1.000000,2184.207862,427 W 21st St Unit Garden,"New York, NY 10011","427 W 21st St Unit GardenNew York, NY 10011",United States,New York,New York County,New York,West 21st Street,"427 W 21st St, New York, NY 10011, USA",40.745882,-74.003398,"Brokered by Douglas Elliman - 575 Madison Ave,..."
4799,Brokered by E Realty International Corp,Condo for sale,598125,2,1.000000,655.000000,91-23 Corona Ave Unit 4G,"Elmhurst, NY 11373","91-23 Corona Ave Unit 4GElmhurst, NY 11373",New York,Queens County,Queens,Flushing,91-23,"91-23 Corona Ave. #4b, Flushing, NY 11373, USA",40.742770,-73.872752,"Brokered by E Realty International Corp, Condo..."


In [50]:
# Create LangChain Documents, saving only essential metadata
documents = []
for idx, row in df.iterrows():
    metadata = {
        "beds": row['beds'],
        "bath": row['bath'],
        "address": row['address'],
        "state": row['state'],
        "propertysqft": row['propertysqft'],
        "price": row['price'],
    }
    doc = Document(page_content=row['text_to_embed'], metadata=metadata)
    documents.append(doc)

In [51]:
documents[0]  # Display the first document to verify

Document(metadata={'beds': 2, 'bath': 2.0, 'address': '2 E 55th St Unit 803', 'state': 'New York, NY 10022', 'propertysqft': 1400.0, 'price': 315000}, page_content='Brokered by Douglas Elliman  -111 Fifth Ave, Condo for sale, Price: 315000$, Beds: 2, Baths: 2.0, Size: 1400.0 sqft, Address: 2 E 55th St Unit 803, Locality: New York, State: New York, NY 10022')

In [52]:
embedding_dim = len(embeddings.embed_query("hello world"))
index = faiss.IndexFlatL2(embedding_dim)

vector_store = FAISS(
    embedding_function=embeddings,
    index=index,
    docstore=InMemoryDocstore(),
    index_to_docstore_id={},
)

In [ ]:
# Add documents to the vector store 
vector_store.add_documents(documents)


  0%|          | 0/4801 [00:00<?, ?it/s]

100%|██████████| 4801/4801 [00:16<00:00, 293.76it/s]


In [61]:
results = vector_store.similarity_search(
    "House in New York with 3 bedrooms and not more than 2 bathrooms between 1000000$ and 3000000$",
    k=2
)
for res in results:
    print(f"* {res.page_content} [{res.metadata}]")

* Brokered by COMPASS, House for sale, Price: 14000000$, Beds: 3, Baths: 2.3738608579684373, Size: 23027.0 sqft, Address: 39 Eldridge St, Locality: New York, State: Manhattan, NY 10002 [{'beds': 3, 'bath': 2.3738608579684373, 'address': '39 Eldridge St', 'state': 'Manhattan, NY 10002', 'propertysqft': 23027.0, 'price': 14000000}]
* Brokered by Carina Realty Inc, House for sale, Price: 1350000$, Beds: 3, Baths: 2.3738608579684373, Size: 2184.207862 sqft, Address: 219 E 115th St, Locality: New York, State: Nyc, NY 10029 [{'beds': 3, 'bath': 2.3738608579684373, 'address': '219 E 115th St', 'state': 'Nyc, NY 10029', 'propertysqft': 2184.207862, 'price': 1350000}]


In [66]:
vector_store.save_local("faiss_index_dir")